This is a starter notebook for the project, you'll have to import the libraries you'll need, you can find a list of the ones available in this workspace in the requirements.txt file in this workspace.

## Step 1: Setting Up the Python Application

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Get the keys from a secret place
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

## Step 2: Generating Real Estate Listings


| Model        | Speed   | Cost      | Quality                                |
|--------------|---------|-----------|----------------------------------------|
| gpt-4o-mini  | Fastest | Cheapest  | Good for simple tasks                  |
| gpt-4o       | Fast    | Mid       | Better reasoning, multimodal           |
| gpt-4-turbo  | Slower  | Expensive | Older, no real advantage over gpt-4o   |

  1. Calls the LLM to generate 10+ listings in the structured format
  2. Saves them to a file so we don't regenerate on every run

In [2]:
prompt = """
    Generate at least 15 diverse real estate listings.
    Vary the neighborhoods, price ranges ($200000-$2000000), styles (urban condo, suburban family home, rural cottage, etc.), and sizes so they contrast meaningfully.
    For each listing, use exactly in json format:

    {
        "neighborhood" : "<name>"
        "price" : <amount>
        "bedrooms" : <n>
        "bathrooms" : <n>
        "house_size_sqft": <n>
        "description": "<2-3 sentence property description>"
        "neighborhood_description": "<2-3 sentence neighborhood description>"
    }
"""

In [3]:
import json

listing_path = "resources/chatgpt_listings.json"
if os.path.exists(listing_path):
    with open(listing_path, "r") as f:
        listings = json.load(f)
    print("Loaded listings from file!\n")
else:
    reponse = llm.invoke([HumanMessage(content=prompt)])
    raw = reponse.content

    raw = raw[raw.index("```json") + 7 : raw.rindex("```")].strip()
    listings = json.loads(raw)

    with open(listing_path, "w") as f:
        json.dump(listings, f, indent=2)
        print("Generated and saved listings")

Loaded listings from file!



In [4]:
print(f"{len(listings)} listings loaded")
print(listings[1])

18 listings loaded
{'neighborhood': 'Greenwood', 'price': 320000, 'bedrooms': 3, 'bathrooms': 2, 'house_size_sqft': 1800, 'description': 'Charming suburban family home with a spacious backyard. Recently updated with a new roof and modern kitchen.', 'neighborhood_description': 'Greenwood is a family-friendly neighborhood with good schools, parks, and a strong sense of community.'}


## Step 3: Storing Listings in a Vector Database

1. initialize ChromaDB and embedder

In [5]:
import chromadb
from langchain_openai import OpenAIEmbeddings

client = chromadb.PersistentClient(path="resources/chroma_db")
collection = client.get_or_create_collection("real_estate")

embedder = OpenAIEmbeddings(model="text-embedding-3-small")
print("ChromaDB collection ready, embedder loaded.")

ChromaDB collection ready, embedder loaded.


2. Load listings and populate the DB (idempotent):

In [6]:
import json

listing_path = "resources/chatgpt_listings.json"
with open(listing_path) as f:
    listings = json.load(f)

# Build a full-text string per listing for embedding
def listing_to_text(l):
    return (
        f"Neighborhood: {l['neighborhood']}\n"
        f"Price: ${l['price']:,}\n"
        f"Bedrooms: {l['bedrooms']}\n"
        f"Bathrooms: {l['bathrooms']}\n"
        f"House Size: {l['house_size_sqft']} sqft\n\n"
        f"Description: {l['description']}\n\n"
        f"Neighborhood Description: {l['neighborhood_description']}"
    )

existing_ids = set(collection.get()["ids"])

for i, listing in enumerate(listings):
    listing_id = f"listing_{i}"
    if listing_id in existing_ids:
        continue # skip if already inserted
    text = listing_to_text(listing)
    embedding = embedder.embed_query(text)
    collection.add(ids=[listing_id], documents=[text], embeddings=[embedding])
    print(f"Added {listing_id} - {listing['neighborhood']}")

print(f"\nCollection now has {collection.count()} listings.")



Collection now has 18 listings.


### A few notes:
  - text-embedding-3-small is cheap (~$0.02/million tokens) and accurate enough for this task
  - The idempotency check (existing_ids) means re-running the cell won't double-insert

## Step 4: Building the User Preference Interface

1. Define buyer preferences

In [7]:
questions = [
  "How big do you want your house to be?",
  "What are 3 most important things for you in choosing this property?",
  "Which amenities would you like?",
  "Which transportation options are important to you?",
  "How urban do you want your neighborhood to be?",
]

answers = [
  "A comfortable three-bedroom house with a spacious kitchen and a cozy living room.",
  "A quiet neighborhood, good local schools, and convenient shopping options.",
  "A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.",
  "Easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads.",
  "A balance between suburban tranquility and access to urban amenities like restaurants and theaters.",
]

2. Build buyer profile string

In [8]:
buyer_profile = "\n".join(f"{q}\n{a}" for q, a in zip(questions, answers))
print(buyer_profile)

How big do you want your house to be?
A comfortable three-bedroom house with a spacious kitchen and a cozy living room.
What are 3 most important things for you in choosing this property?
A quiet neighborhood, good local schools, and convenient shopping options.
Which amenities would you like?
A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.
Which transportation options are important to you?
Easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads.
How urban do you want your neighborhood to be?
A balance between suburban tranquility and access to urban amenities like restaurants and theaters.


## Step 5: Searching Based on Preferences

In [9]:
# semantic search
query_embedding = embedder.embed_query(buyer_profile)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

matched_listings = results["documents"][0]

for i, listing in enumerate(matched_listings):
    print(f"=== Match {i+1} ===")
    print(listing)
    print()

=== Match 1 ===
Neighborhood: Historic District
Price: $785,000
Bedrooms: 4
Bathrooms: 3
House Size: 2600 sqft

Description: Gorgeous historic home with modern renovations and a large front porch. Perfect blend of charm and comfort in a sought-after area.

Neighborhood Description: The Historic District features beautiful architecture, tree-lined streets, and a rich cultural atmosphere.

=== Match 2 ===
Neighborhood: Sunnyvale
Price: $475,000
Bedrooms: 3
Bathrooms: 2
House Size: 1600 sqft

Description: Bright and airy single-family home with a spacious layout. Features a large kitchen and a private yard perfect for outdoor gatherings.

Neighborhood Description: Sunnyvale is a vibrant community with excellent schools and recreational facilities, making it perfect for families.

=== Match 3 ===
Neighborhood: Greenwood
Price: $320,000
Bedrooms: 3
Bathrooms: 2
House Size: 1800 sqft

Description: Charming suburban family home with a spacious backyard. Recently updated with a new roof and mo

## Step 6: Personalizing Listing Descriptions

Uses LLM to rewrite each matched listing's description to appeal to the buyer without changing any factual details.

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

personalize_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def personalize(listing: str, buyer_profile: str) -> str:
    personalize_prompt = f"""
    You are a real estate agent. A buyer has the following preferences:
    
    {buyer_profile}
    
    Rewrite the listing description below to highlight aspects that appeal to this buyer.
    Do NOT change any factual information (price, bedrooms, bathrooms, size, neighborhood name).
    Only rewrite the Description and Neighborhood Description fields.
    
    Listing:
    {listing}
    """

    return personalize_llm.invoke([HumanMessage(content=personalize_prompt)]).content

for i, listing in enumerate(matched_listings):
    print(f"=== Personalized Match {i+1} ===")
    print(personalize(listing, buyer_profile))
    print()

=== Personalized Match 1 ===
**Description:** Welcome to this charming four-bedroom home, ideally situated in the serene Historic District. With a spacious layout that includes a cozy living room and a generously sized kitchen, this home perfectly balances comfort and modern living. Enjoy the tranquility of the neighborhood while benefiting from recent renovations that enhance its historic charm. The inviting large front porch is perfect for relaxing evenings.

**Neighborhood Description:** The Historic District offers a peaceful suburban environment, characterized by its beautiful architecture and tree-lined streets. This family-friendly area boasts excellent local schools, making it perfect for families. You'll find convenient shopping options nearby, ensuring easy access to daily necessities. Plus, with reliable public transportation and bike-friendly roads, commuting and exploring the vibrant urban amenities, including restaurants and theaters, is a breeze.

=== Personalized Match 

## Step 7: Deliverables and Testing

Write 3 different buyer profiles to be tested with the same pipeline, and have meaningful output.

In [11]:
buyer_profiles = [
  {
      "label": "City Dweller",
      "questions": [
          "How big do you want your house to be?",
          "What are 3 most important things for you in choosing this property?",
          "Which amenities would you like?",
          "Which transportation options are important to you?",
          "How urban do you want your neighborhood to be?",
      ],
      "answers": [
          "A compact one or two-bedroom apartment, around 600 to 1200 sqft is plenty.",
          "Walkability, vibrant nightlife, and proximity to dining and entertainment.",
          "Modern kitchen, private balcony or rooftop access, and fast internet.",
          "Walking distance to everything — I don't own a car.",
          "Fully urban — I want cafes, shops, and culture right at my doorstep.",
      ]
  },
  {
      "label": "Growing Family",
      "questions": [
          "How big do you want your house to be?",
          "What are 3 most important things for you in choosing this property?",
          "Which amenities would you like?",
          "Which transportation options are important to you?",
          "How urban do you want your neighborhood to be?",
      ],
      "answers": [
          "A spacious three to four-bedroom home with at least 1800 sqft and a backyard.",
          "Good school district, safe neighborhood, and a large backyard for the kids.",
          "A modern kitchen, a yard for outdoor play, and a two-car garage.",
          "Close to schools and parks, easy highway access for commuting.",
          "Suburban — quiet streets, friendly neighbors, community feel.",
      ]
  },
  {
      "label": "Nature & Luxury Enthusiast",
      "questions": [
          "How big do you want your house to be?",
          "What are 3 most important things for you in choosing this property?",
          "Which amenities would you like?",
          "Which transportation options are important to you?",
          "How urban do you want your neighborhood to be?",
      ],
      "answers": [
          "A large home with at least 2000 sqft, open layouts, and scenic views.",
          "Stunning natural surroundings, high-end finishes, and privacy.",
          "An expansive deck, chef's kitchen, and access to hiking or water activities.",
          "A car is fine — I prefer scenic drives over public transit.",
          "Semi-rural or upscale suburban — close to nature but with premium comforts.",
      ]
  },
]

for profile in buyer_profiles:
    label = profile["label"]
    buyer_profile = "\n".join(f"{q}\n{a}" for q, a in zip(profile["questions"], profile["answers"]))

    print(f"\n{'='*60}")
    print(f"BUYER PROFILE: {label}")
    print('='*60)
    
    query_embedding = embedder.embed_query(buyer_profile)
    results = collection.query(query_embeddings=[query_embedding], n_results=3)
    matched_listings = results["documents"][0]
    
    for i, listing in enumerate(matched_listings):
        print(f"\n--- Match {i+1} (Personalized) ---")
        print(personalize(listing, buyer_profile))


BUYER PROFILE: City Dweller

--- Match 1 (Personalized) ---
**Description:** Embrace urban living in this chic one-bedroom condo, featuring high ceilings and stainless steel appliances that cater to your modern lifestyle. Enjoy your morning coffee or evening relaxation on your private balcony, where you can soak in the stunning city views. With 650 sqft of well-designed space, this condo is perfect for those seeking a compact yet stylish home in an ideal location.

**Neighborhood Description:** Experience the pulse of city life in Downtown, where vibrant nightlife, trendy cafes, and an array of dining options are right at your doorstep. This fully urban neighborhood ensures you're never far from entertainment and culture, making it easy to explore everything the city has to offer—all within walking distance.

--- Match 2 (Personalized) ---
**Description:** Discover this stylish urban loft, perfect for those seeking a compact yet modern living space. With 2 bedrooms and 2 bathrooms, th